In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os 
from tqdm.auto import tqdm


def get_mp4_file_sizes(root_folder):
    video_info_list = []
    
    
    for dirpath, _ , filenames in tqdm(os.walk(root_folder)):
        for filename in filenames:
            if filename.lower().endswith(".mp4"):
                full_path = os.path.join(dirpath, filename)
                
                cap = cv2.VideoCapture(full_path)
                
                if not cap.isOpened():
                    print(f"Error opening video file: {full_path}")
                    continue
                    
                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = cap.get(cv2.CAP_PROP_FPS)
                
                cap.release()
                
                video_info_list.append({"path" : full_path,
                                        "width" : width,
                                        "height" : height,
                                        "frame_count" : frame_count,
                                        "fps" : fps})
                
    return video_info_list

/home/mlewand/anaconda3/envs/vid2e/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
path = "../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos"
os.listdir(path) 


['train', 'test', 'val']

In [20]:
results = get_mp4_file_sizes(os.path.join(path, "train"))

0it [00:00, ?it/s]

In [22]:
results[0]

{'path': '../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/train/VIRAT_S_000202_02_001527_001560/004191.mp4',
 'width': 22,
 'height': 22,
 'frame_count': 63,
 'fps': 30.0}

In [24]:
# serialize the the results to a parquet file

import pandas as pd

df = pd.DataFrame(results)
df.to_parquet("video_info.parquet", index=False)

In [25]:
# load the data 
df_loaded = pd.read_parquet("video_info.parquet")

In [26]:
df_loaded

,path,width,height,frame_count,fps
0,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,22,22,63,30.0
1,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,94,94,57,30.0
2,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,22,22,79,30.0
3,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,84,84,115,30.0
4,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,22,22,150,30.0
...,...,...,...,...,...
16945,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,112,112,64,30.0
16946,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,118,118,64,30.0
16947,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,56,56,40,30.0
16948,../../data/TinyVIRAT-v2/TinyVIRAT_V2/videos/tr...,112,112,64,30.0


In [39]:
from collections import Counter

sizes = [tuple(el) for el in df_loaded[["width", "height"]].to_numpy()]


cnt = Counter(sizes)
cnt = sorted([(v, k) for k, v in cnt.items()])[::-1]

for k, v in cnt:
    print(f"Dimension = {v}, Count = {k}")
    

Dimension = (112, 112), Count = 7288
Dimension = (56, 56), Count = 1667
Dimension = (124, 124), Count = 369
Dimension = (120, 120), Count = 349
Dimension = (128, 128), Count = 327
Dimension = (122, 122), Count = 315
Dimension = (126, 126), Count = 312
Dimension = (40, 40), Count = 296
Dimension = (118, 118), Count = 287
Dimension = (114, 114), Count = 286
Dimension = (116, 116), Count = 280
Dimension = (44, 44), Count = 258
Dimension = (38, 38), Count = 204
Dimension = (110, 110), Count = 194
Dimension = (46, 46), Count = 181
Dimension = (36, 36), Count = 180
Dimension = (66, 66), Count = 175
Dimension = (34, 34), Count = 170
Dimension = (50, 50), Count = 166
Dimension = (70, 70), Count = 153
Dimension = (58, 58), Count = 149
Dimension = (106, 106), Count = 148
Dimension = (104, 104), Count = 145
Dimension = (24, 24), Count = 141
Dimension = (52, 52), Count = 139
Dimension = (48, 48), Count = 138
Dimension = (102, 102), Count = 136
Dimension = (60, 60), Count = 136
Dimension = (28, 28)

In [56]:
def get_names_with_size(df, size):
    return df[df["width"] == size[0]][df["height"] == size[1]]["path"].values

In [57]:
names = get_names_with_size(df_loaded, (112,112))

/tmp/ipykernel_45512/1826951355.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  return df[df["width"] == size[0]][df["height"] == size[1]]["path"].values


In [ ]:
from IPython.display import HTML
from pathlib import Path
import shutil
import subprocess
from pathlib import Path
import time

import time

def convert_and_display_video(idx):

    subprocess.run("rm *.mp4", shell=True)

    input_name = Path(names[idx])
    timestamp = int(time.time() * 1000)  # unique ID for filename
    copied_name = f"video_{idx}_{timestamp}.mp4"
    output_name = f"video_{idx}_{timestamp}_h264.mp4"

    # Copy the file
    shutil.copy(input_name, copied_name)

    # Convert it
    command = [
        "ffmpeg",
        "-i", copied_name,
        "-vcodec", "libx264",
        "-acodec", "aac",
        "-movflags", "+faststart",
        output_name,
        "-y"
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Display it
    display(HTML(f"""
        <video width="640" height="480" controls>
          <source src="{output_name}" type="video/mp4">
          Your browser does not support the video tag.
        </video>
        """))

convert_and_display_video(0)

FileNotFoundError: [Errno 2] No such file or directory: '*mp4'